# 📊 Exploratory Data Analysis — Review Sentiment Analyzer
Visualise the raw and cleaned review data before model training.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from wordcloud import WordCloud
from collections import Counter

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# ── Load data (generate sample if not present) ─────────────────────────────
CLEAN_PATH = '../data/cleaned_reviews.csv'

if os.path.exists(CLEAN_PATH):
    df = pd.read_csv(CLEAN_PATH)
else:
    from scraper.scraper import generate_sample_reviews
    from preprocessing.preprocess import preprocess_dataframe
    raw = pd.DataFrame(generate_sample_reviews(500))
    df = preprocess_dataframe(raw)
    os.makedirs('../data', exist_ok=True)
    df.to_csv(CLEAN_PATH, index=False)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Basic stats ────────────────────────────────────────────────────────────
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nSentiment distribution:')
print(df['sentiment'].value_counts())

In [ ]:
# ── Sentiment distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'positive': '#22c55e', 'neutral': '#f59e0b', 'negative': '#ef4444'}
counts = df['sentiment'].value_counts()

axes[0].bar(counts.index, counts.values,
            color=[colors[s] for s in counts.index], edgecolor='white', linewidth=1.5)
axes[0].set_title('Sentiment Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=counts.index,
            colors=[colors[s] for s in counts.index],
            autopct='%1.1f%%', startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Sentiment Share', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Rating distribution ────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
sns.histplot(df['rating'], bins=20, kde=True, color='#6366f1')
plt.title('Rating Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Star Rating')
plt.tight_layout()
plt.show()

In [ ]:
# ── Review length analysis ─────────────────────────────────────────────────
df['review_len'] = df['review'].str.split().str.len()
df['cleaned_len'] = df['cleaned_text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes,
                           ['review_len', 'cleaned_len'],
                           ['Raw Review Length (words)', 'Cleaned Review Length (words)']):
    for sent, grp in df.groupby('sentiment'):
        ax.hist(grp[col], bins=40, alpha=0.6, label=sent, color=colors[sent])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Word Count')
    ax.legend()

plt.tight_layout()
plt.show()

print(df.groupby('sentiment')[['review_len', 'cleaned_len']].describe())

In [ ]:
# ── Word clouds per sentiment ──────────────────────────────────────────────
cmaps = {'positive': 'Greens', 'neutral': 'YlOrBr', 'negative': 'Reds'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, sent in zip(axes, ['positive', 'neutral', 'negative']):
    texts = ' '.join(df[df['sentiment'] == sent]['cleaned_text'].dropna())
    wc = WordCloud(width=600, height=350, background_color='white',
                   colormap=cmaps[sent], max_words=100, collocations=False).generate(texts or 'empty')
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{sent.title()} Reviews', fontsize=13, fontweight='bold')

plt.suptitle('Word Clouds by Sentiment', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 20 most frequent words per sentiment ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, sent in zip(axes, ['positive', 'neutral', 'negative']):
    tokens = ' '.join(df[df['sentiment'] == sent]['cleaned_text'].dropna()).split()
    top = pd.DataFrame(Counter(tokens).most_common(20), columns=['word', 'freq'])
    ax.barh(top['word'][::-1], top['freq'][::-1], color=colors[sent])
    ax.set_title(f'Top Words — {sent.title()}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# ── Source breakdown ───────────────────────────────────────────────────────
if 'source' in df.columns:
    fig = px.sunburst(df, path=['source', 'sentiment'],
                      color='sentiment',
                      color_discrete_map=colors,
                      title='Reviews by Source & Sentiment')
    fig.show()

In [ ]:
# ── Correlation: rating vs. sentiment label ────────────────────────────────
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='sentiment', y='rating',
            palette=colors, order=['negative', 'neutral', 'positive'])
plt.title('Rating Distribution by Sentiment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()